# 코드 E · 변수 사전 붙여 읽기와 기업 부도 데이터
**핀테크 데이터 분석 : 실제로 해보는 데이터 분석편 #01**  
원본: [fintech-data/Revolution5](https://github.com/fintech-data/Revolution5) `Loading_Data.ipynb` → 이 폴더의 `original_Loading_Data.ipynb`  
해설 페이지: https://adminmrobo.github.io/fintech_data_analysis/hands_on_01.html

> 코드를 몰라도 괜찮아요. 위에서부터 ▶ 실행 버튼만 차례로 누르면 됩니다.

## 이 코드가 하는 일
① 대출 장부의 74개 열 이름 옆에 **뜻풀이 사전**(`loan_data_description.csv`)을 붙여 한 줄씩 출력하는 작은 함수를 만들고, ② 대만 기업 6,819곳의 재무비율 95개와 부도 여부를 담은 **기업 부도 데이터**를 엽니다.

> 비유: 외국어 메뉴판 옆에 번역 쪽지를 붙여 읽는 일, 그리고 회사 건강검진표 6,819장 넘겨보기.

In [1]:
import os, io, urllib.request
import pandas as pd
pd.set_option('display.max_columns', 12); pd.set_option('display.width', 120)

# 데이터 위치: 이 폴더의 data/ 에 있으면 그걸 쓰고, 없으면(Colab 등) GitHub에서 바로 읽습니다.
RAW = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/hands_on/01/E_dictionary_and_bankruptcy/data/'
def load(name, **kw):
    local = os.path.join('data', name)
    src = local if os.path.exists(local) else RAW + urllib.parse.quote(name)
    return pd.read_csv(src, low_memory=False, **kw)
print('준비 완료')

준비 완료


In [2]:
# loan.csv(441MB)는 GitHub 한 파일 한도 때문에 gzip으로 줄인 뒤 세 조각(part0~2)으로 나눠 두었습니다.
# 조각을 이어 붙이면 원래 파일과 똑같습니다.
def load_loan(folder='E_dictionary_and_bankruptcy', **kw):
    parts = []
    for i in range(3):
        local = os.path.join("../D_loan_ledger_check/data", f'loan.csv.gz.part{i}')
        if os.path.exists(local):
            parts.append(open(local, 'rb').read())
        else:
            parts.append(urllib.request.urlopen('https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/hands_on/01/D_loan_ledger_check/data/' + f'loan.csv.gz.part{i}').read())
    return pd.read_csv(io.BytesIO(b''.join(parts)), compression='gzip', low_memory=False, **kw)

In [3]:
loan = load_loan(usecols=None)
loan_status = load('loan_data_description.csv')
print(loan.shape, loan_status.shape)

(887379, 74) (80, 3)


In [4]:
def description(column_name):
    # 열 이름을 사전에서 찾아 뜻을 돌려줍니다
    return loan_status.loc[loan_status.LoanStatNew == column_name].Description.values[0]

def print_description(column_list):
    missing = []
    for column in column_list:
        try:
            print(f'- {column} = {description(column)}')
        except Exception:
            missing.append(column)
    print('\n사전에 없는 열:', missing)

print_description(loan.columns[:10])

- id = A unique LC assigned ID for the loan listing.
- member_id = A unique LC assigned Id for the borrower member.
- loan_amnt = The listed amount of the loan applied for by the borrower. If at some point in time, the credit department reduces the loan amount, then it will be reflected in this value.
- funded_amnt = The total amount committed to that loan at that point in time.
- funded_amnt_inv = The total amount committed by investors for that loan at that point in time.
- term = The number of payments on the loan. Values are in months and can be either 36 or 60.
- int_rate = Interest Rate on the loan
- installment = The monthly payment owed by the borrower if the loan originates.
- grade = LC assigned loan grade
- sub_grade = LC assigned loan subgrade

사전에 없는 열: []


In [5]:
print_description(loan.columns)

- id = A unique LC assigned ID for the loan listing.
- member_id = A unique LC assigned Id for the borrower member.
- loan_amnt = The listed amount of the loan applied for by the borrower. If at some point in time, the credit department reduces the loan amount, then it will be reflected in this value.
- funded_amnt = The total amount committed to that loan at that point in time.
- funded_amnt_inv = The total amount committed by investors for that loan at that point in time.
- term = The number of payments on the loan. Values are in months and can be either 36 or 60.
- int_rate = Interest Rate on the loan
- installment = The monthly payment owed by the borrower if the loan originates.
- grade = LC assigned loan grade
- sub_grade = LC assigned loan subgrade
- emp_title = The job title supplied by the Borrower when applying for the loan.*
- emp_length = Employment length in years. Possible values are between 0 and 10 where 0 means less than one year and 10 means ten or more years. 
- home

In [6]:
bankruptcy = load('company_bankruptcy.csv')
print(bankruptcy.shape)
bankruptcy['Bankrupt?'].value_counts()

(6819, 96)


Bankrupt?
0    6599
1     220
Name: count, dtype: int64

In [7]:
# 열 이름 앞에 빈 값(공백)이 숨어 있어요
print(repr(bankruptcy.columns[1]))
bankruptcy.columns = bankruptcy.columns.str.strip()
print(repr(bankruptcy.columns[1]))

' ROA(C) before interest and depreciation before interest'
'ROA(C) before interest and depreciation before interest'


In [8]:
# 부도 기업 비율
print(round(bankruptcy['Bankrupt?'].mean() * 100, 1), '%')
bankruptcy.groupby('Bankrupt?')[['Debt ratio %', 'Net worth/Assets']].mean().round(3)

3.2 %


,Debt ratio %,Net worth/Assets
Bankrupt?,,
0,0.111,0.889
1,0.187,0.813
